
  # Laboratorio 1 — Implementando un Algoritmo Genético para Optimizar  una Parábola

  **Materia:** Inteligencia Artificial — FP-UNA
  **Profesor:** Christian von Lücken
  **Alumno:** David Gómez

  **Objetivo:** encontrar los parámetros `(a, b, c)` de la parábola `y =
  a·x² + b·x + c`
  que produzcan la curva **más plana posible hacia arriba** (`a > 0`),
  usando un
  algoritmo genético implementado desde cero.

 ## Parte 1 — Lectura y comprensión teórica


 ### 1.1 ¿Qué es un algoritmo genético y en qué se inspira?
 

Un algoritmo genético (GA) es una **metaheurística de búsqueda y optimización, poblacional y estocástica**: resuelve un problema haciendo evolucionar un conjunto de soluciones candidatas, en lugar de construir una única solución paso a paso. Pertenece a la familia de las heurísticas tal como se las definió en clase, o sea técnicas que buscan soluciones buenas(pero que pueden no ser las mejores) a un costo computacional razonable, sin garantizar factibilidad ni optimalidad, pero no pueden dererminar qué tan cerca quedo el resultado del valor optimo. Fue desarrollado por **John H. Holland** a principios de la década de 1960 con el nombre de "planes reproductivos y adaptativos", en un intento por hacer que las computadoras aprendieran imitando el proceso de la evolución.

Su funcionamiento es un ciclo que se repite: se parte de una población inicial de soluciones codificadas, se le asigna a cada una un valor de aptitud (*fitness*) que mide su calidad, y se construye la generación siguiente aplicando selección, cruzamiento y mutación. Al repetir el ciclo, la aptitud de la población tiende a mejorar.

La inspiración es el **neo-darwinismo**, que explica toda la vida en el planeta mediante cuatro procesos: **reproducción, mutación, competencia y selección**. Un algoritmo genético es, literalmente, una simulación computacional de esos cuatro procesos. La referencia de fondo es Darwin (1858): la evolución se origina por cambios aleatorios en características hereditarias combinados con un proceso de selección natural. La distinción con Lamarck es importante y no es un detalle histórico: el individuo no mejora durante su vida para después transmitir esa mejora, sino que la variación aparece **al azar** y es la selección la que la filtra. Por eso en un GA la mutación es ciega al objetivo y toda la presión hacia la mejora la ejerce la selección.

De la biología se toma también el vocabulario: el **cromosoma** (o individuo) es una solución candidata codificada; el **gen** es cada una de sus componentes; el **alelo**, el conjunto de valores que un gen puede tomar; el **genotipo** es la solución codificada y el **fenotipo**, la solución ya decodificada en el dominio del problema. En este laboratorio el individuo es la terna `(a, b, c)` con los coeficientes de la parábola.

### 1.2 Población, función de fitness, selección, crossover y mutación

**Población.** Conjunto de soluciones candidatas que el algoritmo mantiene y hace evolucionar en paralelo. Que sean varias a la vez es lo que distingue a un GA de una búsqueda local, y lo que le permite muestrear distintas regiones del espacio al mismo tiempo.
*En este laboratorio:* `population_size` ternas `(a, b, c)` de números reales, generadas al azar con `random.uniform(lower_bound, upper_bound)` dentro del rango `[-50, 50]`. La codificación es directa en punto flotante y no binaria, así que acá el genotipo coincide con el fenotipo.

**Función de fitness (aptitud).** Asigna a cada individuo un número que mide qué tan buena es la solución que representa. Es el único componente que depende del problema: todo el resto del algoritmo es genérico.
*En este laboratorio:* mide qué tan plana es la parábola. Calcula el vértice en `x = -b/(2a)`, evalúa cuánto sube la curva desde el vértice hasta `x = -1` y hasta `x = 1`, suma esas dos subidas (`curviness`) y devuelve el valor **negado**, porque el GA maximiza y lo que se busca es minimizar la curvatura. Si `a <= 0` devuelve `-inf`, descartando de entrada las parábolas que abren hacia abajo. Nótese que `c` no influye en el resultado: desplazar la parábola verticalmente no la hace más ni menos plana.

**Selección.** Decide qué individuos se reproducen, dando más probabilidad a los de mejor aptitud. La intensidad de ese sesgo es la **presión selectiva**: mucha presión acelera la convergencia pero aumenta el riesgo de convergencia prematura; poca presión hace que el algoritmo casi no progrese.
*En este laboratorio:* selección por **torneo de tamaño 3** — se toman tres individuos al azar y pasa el de mejor fitness, repitiendo hasta completar la población. La alternativa clásica es la **ruleta**, donde la probabilidad de ser elegido es proporcional al fitness.

**Crossover (cruzamiento).** Combina el material genético de dos padres para producir hijos. Es el mecanismo de **explotación**: no crea material nuevo, recombina el que ya está presente en la población.
*En este laboratorio:* cruzamiento **aritmético**. Se sortea un `alpha` en `[0, 1)` y los hijos resultan de las combinaciones `alpha·p1 + (1-alpha)·p2` y `alpha·p2 + (1-alpha)·p1`. Cada hijo cae sobre el segmento que une a los dos padres, lo que explica por qué la población tiende a contraerse si no interviene la mutación.

**Mutación.** Altera genes al azar con cierta probabilidad. Es el mecanismo de **exploración** o diversificación: es la única fuente de material genético verdaderamente nuevo y, por lo tanto, lo único capaz de sacar a la población de una región donde quedó estancada.
*En este laboratorio:* cada gen se modifica con probabilidad `mutation_rate` sumándole un valor uniforme en `[-1, 1]`, y a continuación se lo recorta para que no se salga de `[lower_bound, upper_bound]`.

A estos cinco componentes el código del tutorial le agrega **elitismo**: el mejor individuo de cada generación se copia sin modificaciones a la generación siguiente, de manera que la mejor solución encontrada nunca se pierde.

### 1.3 ¿Por qué se usa un algoritmo genético para resolver problemas de optimización?

Existen técnicas clásicas muy eficaces cuando el problema tiene la estructura adecuada: el método Simplex para optimización lineal, el gradiente conjugado para la no lineal, la programación dinámica o *branch & bound* para construir soluciones parcialmente. Un algoritmo genético no compite con ellas en ese terreno; se recurre a él justamente cuando esas condiciones no se cumplen.

Un problema se vuelve difícil, en el sentido visto en clase, cuando el número de soluciones posibles es tan grande que una búsqueda exhaustiva resulta inviable el caso típico es el problema del viajante, donde con 50 ciudades hay del orden de 10²⁵ recorridos posibles, cuando la función de evaluación es ruidosa o varía con el tiempo, lo que obliga a trabajar con un conjunto de soluciones en lugar de una sola, o cuando las restricciones son tantas que incluso construir una solución factible es complicado.

En esas condiciones un GA aporta tres ventajas concretas:

- **No necesita derivadas** ni que la función objetivo sea continua o diferenciable. Alcanza con poder evaluarla, es decir, sirve para optimización de caja negra.
- **Trabaja con una población**, de modo que muestrea varias regiones del espacio simultáneamente. Eso lo hace menos propenso a quedar atrapado en un óptimo local que una búsqueda que avanza siguiendo una única solución.
- **Es independiente del dominio.** Lo único específico del problema son la codificación y la función de fitness; el resto —selección, cruzamiento, mutación— es idéntico para una parábola que para el problema del viajante.

La contrapartida es que un GA es una heurística: entrega soluciones buenas a un costo computacional razonable, pero no garantiza alcanzar el óptimo global ni permite saber, en general, a qué distancia del óptimo quedó. Este laboratorio lo ilustra con claridad: la curvatura se minimiza cuando `a` tiende a cero por derecha y `b` tiende a cero, es decir sobre el borde de la región factible (`a > 0`), de modo que el algoritmo puede acercarse indefinidamente pero nunca llegar exactamente.

## Parte 2 — Implementación guiada

In [33]:
import random
import numpy as np
import matplotlib.pyplot as plt
from prettytable import PrettyTable

random.seed(42) # misma semilla para que podamos tener los mismos resultados y evaluar el resultado

In [34]:
def create_initial_population(size, lower_bound, upper_bound):
    population = []
    for _ in range(size):
        individual = (random.uniform(lower_bound, upper_bound), 
                      random.uniform(lower_bound, upper_bound),
                      random.uniform(lower_bound, upper_bound)
                      )
        population.append(individual)
    return population

In [35]:
def fitness_function(params):
    a, b, c = params
    if a <= 0:
        return -float('inf') # penalizamos las parabolas que se abren hacia abajo

    vertexX = -b / (2 * a)
    vertexY = a * vertexX**2 + b *vertexX + c

    yLeft = a * (-1) ** 2 + b * (-1) + c # y en x = -1
    yRight = a * (1) ** 2 + b * (1) + c # y en x = 1

    curviness = abs(yLeft - vertexY) + abs(yRight - vertexY) 
    return -curviness

In [36]:
population = create_initial_population(5, -50, 50)
for ind in population:
    print(ind)

for p in [(0.5, 0, 3), (0.5, 0, -40), (2, 3, 1), (0.1, 0, 0), (5, 0, 0), (-0.5, 0, 3), (0,1, 1)]:
    print(p, "->", fitness_function(p))

(13.942679845788376, -47.498924477733304, -22.497068163088073)
(-27.678926185117724, 23.647121416401234, 17.669948742291126)
(39.217956770484534, -41.30611673705839, -7.807818031472955)
(-47.020278056192964, -28.136202519639664, 0.5355288103362383)
(-47.34640303161364, -30.11623493133515, 14.988443777952327)
(0.5, 0, 3) -> -1.0
(0.5, 0, -40) -> -1.0
(2, 3, 1) -> -6.25
(0.1, 0, 0) -> -0.2
(5, 0, 0) -> -10.0
(-0.5, 0, 3) -> -inf
(0, 1, 1) -> -inf


In [37]:
def selection(population, fitnesses, tournament_size=3):
    selected = []
    for _ in range(len(population)):
        tournament = random.sample(list(zip(population, fitnesses)), tournament_size)
        winner = max(tournament, key=lambda x: x[1] )[0]
        selected.append(winner)
    return selected


In [ ]:
selectioned = selection(population, [fitness_function(p) for p in population])
for ind in selectioned:
    print(ind, "->", fitness_function(ind))

(13.942679845788376, -47.498924477733304, -22.497068163088073) -> -108.79332901653116
(39.217956770484534, -41.30611673705839, -7.807818031472955) -> -100.18864392772366
(-27.678926185117724, 23.647121416401234, 17.669948742291126) -> -inf
(39.217956770484534, -41.30611673705839, -7.807818031472955) -> -100.18864392772366
(39.217956770484534, -41.30611673705839, -7.807818031472955) -> -100.18864392772366


In [ ]:
def crossover(parent1, parent2): # el cruzamiento
    alpha = random.random()
    child1 = tuple(alpha * p1 + (1 - alpha) * p2 for p1, p2 in zip(parent1, parent2))
    child2 = tuple(alpha * p2 + (1 - alpha) * p1 for p1, p2 in zip(parent1, parent2))
    return child1, child2

def mutate(individual, mutation_rate, lower_bound, upper_bound):
    individual = list(individual)
    for i in range(len(individual)):
        if random.random() < mutation_rate:
            mutation_amount = random.uniform(-1, 1)
            individual[i] += mutation_amount
            individual[i] = max(min(individual[i], upper_bound), lower_bound)
    return tuple(individual)


In [ ]:
for paren1, parent2 in zip(selectioned[::2], selectioned[1::2]):
    child1, child2 = crossover(paren1, parent2)
    print("Parents:", paren1, parent2)
    print("Children:", child1, child2)
    print("mutated Children:", mutate(child1, 0.1, -10, 10), mutate(child2, 0.1, -10, 10))